Groundwater | Transport Modeling Track

# Step 4: Model Implementation — Heat Transport with Transient Seasonal Forcing (MODFLOW 6 GWE)

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → **4-Build** → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Steps 1–3, you defined clear objectives for heat transport, built a perceptual model with thermal parameters ($R \approx 3.2$, $\alpha_L$ = 5–20 m, Limmat at 12.5 °C), and learned the GWE package structure in MODFLOW 6. Now we **build the numerical heat transport model** — loading the existing flow model and coupling it to a GWE model that simulates temperature evolution in the Limmat Valley aquifer using **measured monthly river and air temperatures** spanning the full available record.

| | |
|---|---|
| **Time** | ~90–120 min core + 10 min optional |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Load** an existing flow model and extract its grid, parameters, and boundary conditions
2. **Rebuild** a GWF model with temperature auxiliary variables for SSM coupling
3. **Apply transient temperature forcing** using measured monthly river and air temperatures
4. **Implement** GWE packages (EST, CND, ADV, SSM, IC, OC) for heat transport
5. **Couple** GWF and GWE models in a single MODFLOW 6 simulation
6. **Interpret** seasonal temperature dynamics and energy balance

## Prerequisites

Before starting this notebook, you should have:
- **Completed the flow track** — especially [04f_model_implementation.ipynb](../flow/04f_model_implementation.ipynb)
- **Completed Transport Steps 1–3** — [1_model_goal](01t_model_goal.ipynb), [2_perceptual_model](02t_perceptual_model.ipynb), [3_modflow_transport](03t_modflow_transport.ipynb)
- Familiarity with GWE packages (EST, CND, ADV, SSM)

> **Where does data go?** Files downloaded in this notebook are saved to `~/applied_groundwater_modelling_data/`. The download functions print the exact path each time.

---

## Introduction

In the flow track (NB4), you built a groundwater flow model from scratch — creating the grid, defining boundary conditions, and running a steady-state simulation. For transport, we take a different approach: we **load the existing flow model** and extend it with temperature.

Why not just add GWE packages to the existing simulation? Because the GWF stress packages (RIV, CHD, WEL, RCH) need an `auxiliary='TEMPERATURE'` column that the original model doesn't have. We therefore:
1. Load the flow model and extract all its data
2. Rebuild the GWF model with temperature auxiliaries
3. Add the GWE model alongside it
4. Run both models coupled in one simulation

## 1 — Setup

In [ ]:
# Setup - import required libraries
import sys
import os
import shutil
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*GeoSeries.*')

# Scientific computing
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# FloPy - MODFLOW 6 GWF
import flopy
from flopy.mf6 import MFSimulation, ModflowTdis, ModflowIms
from flopy.mf6 import ModflowGwf, ModflowGwfdisv
from flopy.mf6 import ModflowGwfnpf, ModflowGwfic, ModflowGwfoc
from flopy.mf6 import ModflowGwfrcha, ModflowGwfchd, ModflowGwfwel, ModflowGwfriv

# FloPy - MODFLOW 6 GWE
from flopy.mf6 import ModflowGwe, ModflowGwedisv
from flopy.mf6 import ModflowGweest, ModflowGwecnd, ModflowGweadv
from flopy.mf6 import ModflowGwessm, ModflowGweesl, ModflowGweic, ModflowGweoc
from flopy.mf6 import ModflowGwfgwe

# Course utilities
sys.path.insert(0, '../../_SUPPORT/src')
sys.path.insert(0, '../../_SUPPORT/src/scripts/scripts_exercises')

from data_utils import download_named_file, get_default_data_folder
from model_io_utils import load_base_simulation
from plot_utils import quick_model_plot

# Checkpoint utilities
from shared_functions import check_task_with_solution, create_multiple_choice

print(f'FloPy version: {flopy.__version__}')

In [ ]:
# Define workspace paths
DATA_DIR = get_default_data_folder()
FLOW_WS = os.path.join(DATA_DIR, 'notebook4_model')
TRANSPORT_WS = os.path.join(DATA_DIR, 'transport_notebook4_model')

# ── Skip/rerun control ──────────────────────────────────────
# Set FORCE_RERUN = True to rebuild and re-run everything from scratch.
# When False, existing simulation results are reused (saves ~5 min).
FORCE_RERUN = False

ucn_cached = os.path.join(TRANSPORT_WS, 'gwe_limmat.ucn')
if FORCE_RERUN or not os.path.exists(ucn_cached):
    if os.path.exists(TRANSPORT_WS):
        shutil.rmtree(TRANSPORT_WS)
    os.makedirs(TRANSPORT_WS)
    print(f'Transport workspace: {TRANSPORT_WS} (fresh)')
else:
    print(f'Transport workspace: {TRANSPORT_WS} (cached results found — skipping rebuild)')
    print(f'  Set FORCE_RERUN = True above to rebuild from scratch')

print(f'Data directory: {DATA_DIR}')
print(f'Flow model workspace: {FLOW_WS}')

---

## 2 — Load Flow Model

We load the flow model built in the flow track (NB4). This model already has a calibrated K field, river-aligned DISV grid, and all boundary conditions. We extract everything we need to rebuild it with temperature auxiliaries.

In [ ]:
# Load flow simulation
sim_flow = load_base_simulation(FLOW_WS)
gwf_flow = sim_flow.get_model()

# --- Extract grid ---
disv_flow = gwf_flow.get_package('disv')
ncpl = disv_flow.ncpl.get_data()
gridprops = {
    'ncpl': ncpl,
    'nvert': disv_flow.nvert.get_data(),
    'vertices': disv_flow.vertices.get_data(),
    'cell2d': disv_flow.cell2d.get_data(),
}
modelgrid = gwf_flow.modelgrid

# --- Extract geometry ---
model_top = disv_flow.top.get_data()
model_bottom = disv_flow.botm.get_data()
idomain = disv_flow.idomain.get_data()

# --- Extract hydraulic properties ---
npf_flow = gwf_flow.get_package('npf')
k_array = npf_flow.k.get_data()

# --- Extract boundary condition data ---
riv_spd_raw = gwf_flow.get_package('riv').stress_period_data.get_data(0)
chd_spd_raw = gwf_flow.get_package('chd').stress_period_data.get_data(0)
wel_spd_raw = gwf_flow.get_package('wel').stress_period_data.get_data(0)
recharge_array = gwf_flow.get_package('rcha').recharge.get_data(0)

# --- Extract initial heads from the solved model ---
hds_path = os.path.join(FLOW_WS, f'{gwf_flow.name}.hds')
initial_heads = flopy.utils.HeadFile(hds_path).get_data()[0]

# Summary
print(f'\nModel extracted:')
print(f'  Grid: {ncpl} DISV cells, 1 layer')
print(f'  K range: {k_array.min():.1f} to {k_array.max():.1f} m/day')
print(f'  RIV entries: {len(riv_spd_raw)}')
print(f'  CHD entries: {len(chd_spd_raw)}')
print(f'  WEL entries: {len(wel_spd_raw)}')
print(f'  Head range: {initial_heads.min():.1f} to {initial_heads.max():.1f} m')

### 2.1 — Vertical Discretisation

The flow model uses a single layer, which is adequate for head calibration. For heat transport, vertical resolution can matter: warm river infiltrate enters the top of the aquifer and forms a thin plume that is diluted over the full column in a single-layer model. Splitting into multiple layers concentrates river leakage in the top layer and produces more realistic seasonal temperature amplitudes.

> **Note:** By default this notebook uses `nlay = 1` (single-layer) for fast run times. You can change `nlay = 3` in the next cell to explore the effect of vertical resolution. With 3 layers (15% / 30% / 55% of thickness), the top layer captures the concentrated thermal signal while deeper layers remain closer to background temperature.

**Layer fractions (when nlay = 3):**
| Layer | Fraction | Role |
|-------|----------|------|
| 0 (top) | 15% | Receives river infiltrate, strongest thermal signal |
| 1 (middle) | 30% | Transition zone |
| 2 (bottom) | 55% | Deep aquifer, near-background temperature |

**River rbot constraint:** At river cells, the top layer must extend below the riverbed bottom (`rbot`) with at least 0.5 m clearance, otherwise the RIV package would reference a cell that is too thin. Where needed, we deepen layer 0 at the expense of layer 1.

In [ ]:
# ============================================================
# Vertical discretisation — set nlay to control layer count
# ============================================================
nlay = 1  # Set to 1 for single-layer, 3 for multi-layer

# Flatten geometry arrays
top_1d = model_top.flatten() if model_top.ndim > 1 else model_top
bot_1d = model_bottom.flatten() if model_bottom.ndim > 1 else model_bottom
thickness_1d = np.maximum(top_1d - bot_1d, 0.0)

if nlay == 1:
    # Single layer: use original geometry directly
    botm_layers = bot_1d.reshape(1, -1)
else:
    # Multi-layer split (15% / 30% / 55%)
    layer_fracs = np.array([0.15, 0.30, 0.55])
    cum_fracs = np.cumsum(layer_fracs)
    botm_layers = np.zeros((nlay, ncpl))
    for lay in range(nlay):
        botm_layers[lay] = top_1d - cum_fracs[lay] * thickness_1d

    # River rbot constraint: layer 0 must extend below rbot - 0.5 m
    riv_data = riv_spd_raw
    n_rbot_adjusted = 0
    for entry in riv_data:
        cell_id = entry['cellid']
        cell_idx = cell_id[1] if isinstance(cell_id, tuple) else cell_id
        rbot = entry['rbot']
        required_bot0 = rbot - 0.5
        if botm_layers[0, cell_idx] > required_bot0:
            botm_layers[0, cell_idx] = required_bot0
            remaining = required_bot0 - bot_1d[cell_idx]
            if remaining > 1.0:
                botm_layers[1, cell_idx] = required_bot0 - 0.35 * remaining
            else:
                botm_layers[1, cell_idx] = required_bot0 - 0.5 * max(remaining, 0.1)
            n_rbot_adjusted += 1

    # Enforce strict layer ordering with minimum thickness
    min_layer_thick = 0.1
    for j in range(ncpl):
        for lay in range(nlay):
            upper = top_1d[j] if lay == 0 else botm_layers[lay - 1, j]
            botm_layers[lay, j] = min(botm_layers[lay, j], upper - min_layer_thick)

# --- Build nlay-layer arrays ---
idomain_1d = idomain.flatten() if idomain.ndim > 1 else idomain.copy()
idomain_nd = np.tile(idomain_1d, (nlay, 1)).astype(np.int32)

if nlay > 1:
    # Deactivate layers that extend below aquifer base
    for lay in range(nlay):
        below_base = botm_layers[lay] < bot_1d - 0.01
        idomain_nd[lay, below_base] = 0

k_1d = k_array.flatten() if k_array.ndim > 1 else k_array
k_3d = np.tile(k_1d, (nlay, 1))
k33 = k_3d * 0.1

heads_1d = initial_heads.flatten() if initial_heads.ndim > 1 else initial_heads
initial_heads_3d = np.tile(heads_1d, (nlay, 1))

# Summary
for lay in range(nlay):
    lay_top = top_1d if lay == 0 else botm_layers[lay - 1].flatten()
    lay_thick = lay_top - botm_layers[lay].flatten()
    print(f'  Layer {lay}: thickness {lay_thick.min():.1f} to {lay_thick.max():.1f} m (mean {lay_thick.mean():.1f} m)')
print(f'  Total cells: {nlay} x {ncpl} = {nlay * ncpl}')

In [ ]:
# Quick verification: plot heads from the flow model
boundary_path = download_named_file(name='model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_path)

fig, ax = plt.subplots(figsize=(12, 10))
quick_model_plot(
    modelgrid, initial_heads.flatten(), boundary_gdf=boundary_gdf,
    title='Flow Model — Simulated Heads (Starting Point)',
    cmap='Blues', colorbar_label='Head (m a.s.l.)', ax=ax
)
plt.tight_layout()
plt.show()

---

## 3 — Transport Parameters

These values come from the perceptual model (NB2) and the GWE package mapping (NB3). We define all thermal parameters here as Python variables, then use them when building the GWE model.

In [ ]:
# ============================================================
# THERMAL PARAMETERS
# ============================================================

# --- EST inputs (Energy Storage and Transfer) ---
n_e = 0.20          # Effective porosity (transport, slightly lower than total porosity)
rho_s = 2650.0      # Solid grain density [kg/m³] (quartz/feldspar gravels)
c_s = 880.0         # Solid heat capacity [J/(kg·K)] (typical for silicate minerals)
rho_w = 1000.0      # Water density [kg/m³]
c_w = 4184.0        # Water heat capacity [J/(kg·K)]

# --- CND inputs (Conduction-Dispersion) ---
# Component thermal conductivities (same as NB2)
lambda_w = 0.60     # Thermal conductivity of water [W/(m·K)]
lambda_s = 2.0      # Thermal conductivity of solid [W/(m·K)]
# MODFLOW 6 CND uses arithmetic mixing: lambda_bulk = n*lambda_w + (1-n)*lambda_s
# With these values: lambda_bulk ≈ 0.20*0.60 + 0.80*2.0 = 1.72 W/(m·K)
# NB2 used geometric mean giving ~1.57 — both within the physical range for saturated gravel
lambda_bulk = n_e * lambda_w + (1 - n_e) * lambda_s

# Convert to daily units for MODFLOW (model time = days)
# 1 W/(m·K) = 1 J/(s·m·K) = 86400 J/(d·m·K)
ktw = lambda_w * 86400   # Thermal conductivity of water [J/(d·m·K)]
kts = lambda_s * 86400   # Thermal conductivity of solid [J/(d·m·K)]

# Dispersivity
alpha_L = 10.0      # Longitudinal dispersivity [m]
alpha_T = 1.0       # Transverse dispersivity [m] (alpha_L / 10)

# --- Temperatures (annual means — overridden by monthly data for RIV/RCHA) ---
T_background = 10.5  # Background groundwater temperature [°C]
T_limmat = 12.5      # River Limmat annual mean [°C] (overridden by monthly data)
T_sihl = 11.1        # River Sihl annual mean [°C] (overridden by monthly data)
T_recharge = 9.8     # Areal recharge annual mean [°C] (overridden by monthly data)
T_lateral = 10.5     # Lateral inflow temperature [°C] (background, constant)
T_chd = 10.5         # CHD outflow temperature [°C] (background, constant)

# --- Compute thermal retardation factor ---
R_thermal = (n_e * rho_w * c_w + (1 - n_e) * rho_s * c_s) / (n_e * rho_w * c_w)

# --- ESL: Energy Source Loading ---
q_geothermal = 0.070  # Geothermal heat flux [W/m²]
q_urban = 1.5         # Urban heat island flux [W/m²]
urban_x_min = 2.6835e6  # Approximate eastern boundary of urban zone [m, LV95]

# Convert to daily energy units for MODFLOW (model time = days)
# 1 W/m² = 1 J/(s·m²) = 86400 J/(d·m²)
q_geo_daily = q_geothermal * 86400   # [J/(d·m²)]
q_urban_daily = q_urban * 86400      # [J/(d·m²)]

print('Thermal parameters:')
print(f'  Porosity (n_e):         {n_e}')
print(f'  Solid density:          {rho_s} kg/m³')
print(f'  Solid heat capacity:    {c_s} J/(kg·K)')
print(f'  Bulk λ:                 {lambda_bulk:.2f} W/(m·K)')
print(f'  λ_water (daily):        {ktw:.0f} J/(d·m·K)')
print(f'  λ_solid (daily):        {kts:.0f} J/(d·m·K)')
print(f'  α_L / α_T:             {alpha_L} / {alpha_T} m')
print(f'  Retardation R:          {R_thermal:.2f}')
print(f'  → Thermal front ≈ 1/{R_thermal:.1f} of water velocity')

### Parameter Summary

| Parameter | Description | GWE Package | Role |
|-----------|-------------|-------------|------|
| $n_e$ = 0.20 | Effective porosity | **EST** | Pore velocity, thermal retardation |
| $\rho_s$ = 2650 kg/m³ | Solid density | **EST** | Thermal retardation |
| $c_s$ = 880 J/(kg·K) | Solid heat capacity | **EST** | Thermal retardation |
| $\lambda_w$ = 0.60 W/(m·K) | Water thermal conductivity | **CND** (`ktw`) | Conduction |
| $\lambda_s$ = 2.0 W/(m·K) | Solid thermal conductivity | **CND** (`kts`) | Conduction |
| $\alpha_L$ = 10 m | Longitudinal dispersivity | **CND** (`alh`) | Mechanical dispersion |
| $\alpha_T$ = 1 m | Transverse dispersivity | **CND** (`ath1`) | Mechanical dispersion |
| $T_0$ = 10.5 °C | Background temperature | **IC** | Initial condition |
| $T_{Limmat}$ = 12.5 °C | Limmat River temperature | **SSM** (via RIV aux) | Warm thermal input |
| $T_{Sihl}$ = 11.1 °C | Sihl River temperature | **SSM** (via RIV aux) | Near-background input |
| $T_{recharge}$ = 9.8 °C | Recharge temperature | **SSM** (via RCH aux) | Cool surface input |

<details>
<summary><strong>Optional: GWT (solute) equivalent parameters</strong> (click to expand)</summary>

If you were building a solute transport model (GWT) instead of heat (GWE), the equivalent parameters would be:

| GWE parameter | GWT equivalent | Notes |
|---------------|----------------|-------|
| $n_e$, $\rho_s$, $c_s$ (EST) | $n_e$, $\rho_b$, $K_d$ (MST) | Thermal retardation → sorption retardation |
| $\lambda_w$, $\lambda_s$ (CND) | $D_m^*$ (DSP) | Thermal conductivity → molecular diffusion |
| $\alpha_L$, $\alpha_T$ (CND) | $\alpha_L$, $\alpha_T$ (DSP) | **Identical** — mechanical dispersion is the same |
| Temperature auxiliaries | Concentration auxiliaries | Same SSM coupling mechanism |

**Key insight:** $\alpha_L$ and $\alpha_T$ transfer directly from heat to solute models — this is why calibrating dispersivity with heat is so useful.

</details>

In [ ]:
# Checkpoint 1 — Thermal Retardation Factor
# Calculate R using the parameters above
check_task_with_solution('task_t04_checkpoint_1')

### Load Temperature Forcing Data

Instead of using constant annual-mean temperatures, we use **measured monthly temperature data** for the river and recharge boundaries. This creates a transient simulation with monthly stress periods that captures the seasonal temperature cycle — summer peaks in the rivers (>20 °C) and winter lows (~5 °C for the Limmat, ~2 °C for the Sihl).

We use the **full available record** so the thermal signal has enough time to propagate through the aquifer (thermal retardation $R \approx 3.2$ means heat moves ~3× slower than water — short simulations barely warm the aquifer).

The data sources:
- **River Limmat:** monthly mean water temperature from station 0578 (downstream of Lake Zurich)
- **River Sihl:** monthly mean water temperature from station 0577
- **Air temperature:** daily measurements from Fluntern (Zurich), aggregated to monthly means for recharge temperature

In [ ]:
# Load measured temperature data
import calendar

# --- River Limmat: monthly mean water temperature ---
limmat_csv = os.path.join(DATA_DIR, 'river_temperature_limmat_monthly.csv')
df_limmat = pd.read_csv(limmat_csv)

# --- River Sihl: monthly mean water temperature ---
sihl_csv = os.path.join(DATA_DIR, 'river_temperature_sihl_monthly.csv')
df_sihl = pd.read_csv(sihl_csv)

# --- Air temperature: daily → monthly means for recharge ---
air_csv = os.path.join(DATA_DIR, 'air_temperature_fluntern.csv')
df_air = pd.read_csv(air_csv, parse_dates=['date'])
df_air['year'] = df_air['date'].dt.year
df_air['month'] = df_air['date'].dt.month
air_monthly_all = df_air.groupby(['year', 'month'])['temperature'].mean().reset_index()

# Find common complete years across all three datasets
lim_complete = set(df_limmat.groupby('year').filter(lambda x: len(x) == 12)['year'])
sih_complete = set(df_sihl.groupby('year').filter(lambda x: len(x) == 12)['year'])
air_complete = set(air_monthly_all.groupby('year').filter(lambda x: len(x) == 12)['year'])
common_years = sorted(lim_complete & sih_complete & air_complete)
year_start, year_end = common_years[0], common_years[-1]

# Filter to common complete years
df_limmat = df_limmat[df_limmat['year'].isin(common_years)].sort_values(['year', 'month'])
df_sihl = df_sihl[df_sihl['year'].isin(common_years)].sort_values(['year', 'month'])
T_limmat_monthly = df_limmat['mean'].values
T_sihl_monthly = df_sihl['mean'].values

# Air temperature → recharge temperature (clamped to >= 0.5 °C: winter recharge arrives as ~0 °C snowmelt)
air_filtered = air_monthly_all[air_monthly_all['year'].isin(common_years)].sort_values(['year', 'month'])
T_recharge_monthly = np.maximum(air_filtered['temperature'].values, 0.5)

nper = len(T_limmat_monthly)
nyears = len(common_years)

print(f'Temperature forcing loaded: {nper} months ({year_start}–{year_end}, {nyears} years)')
print(f'  Limmat:   {T_limmat_monthly.min():.1f} to {T_limmat_monthly.max():.1f} °C (mean {T_limmat_monthly.mean():.1f} °C)')
print(f'  Sihl:     {T_sihl_monthly.min():.1f} to {T_sihl_monthly.max():.1f} °C (mean {T_sihl_monthly.mean():.1f} °C)')
print(f'  Recharge: {T_recharge_monthly.min():.1f} to {T_recharge_monthly.max():.1f} °C (mean {T_recharge_monthly.mean():.1f} °C)')

In [ ]:
# Plot forcing time series
month_labels = pd.date_range(f'{year_start}-01', periods=nper, freq='MS')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(month_labels, T_limmat_monthly, '-', color='royalblue', label='Limmat (river)', linewidth=0.8)
ax.plot(month_labels, T_sihl_monthly, '-', color='steelblue', label='Sihl (river)', linewidth=0.8)
ax.plot(month_labels, T_recharge_monthly, '-', color='seagreen', label='Recharge (air T, clamped)', linewidth=0.8)
ax.axhline(T_background, color='gray', linestyle='--', alpha=0.7, label=f'Background ({T_background} °C)')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.set_title(f'Temperature Forcing Data — Monthly Means ({year_start}–{year_end})')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 4 — Time Discretisation

The flow model is steady-state, but heat transport is **transient** — we need to track how the temperature field evolves over time. With monthly temperature forcing, we use **one stress period per calendar month** across the full data record, with ~5-day time steps per period.

Because thermal retardation slows the heat front to ~1/3 of the water velocity, the aquifer needs **decades** of forcing before the seasonal temperature signal fully develops — this is why we use the entire available record rather than just a few years.

Recall from NB3 the **Courant number** criterion:

$$Cr = \frac{v \cdot \Delta t}{\Delta x}$$

Using our Limmat Valley values: $v \approx 2.3$ m/d (domain-average from NB2), $\Delta t = 5$ days, $\Delta x \approx 25$ m (river cell size). Near the rivers, however, velocities can reach **~11 m/d** (NB2, near the Hardhof wellfield) — the checkpoint below asks you to evaluate the Courant number for this near-river velocity.

In [ ]:
# Time discretisation — monthly stress periods from calendar months
periods = []
for year in common_years:
    for month in range(1, 13):
        ndays = calendar.monthrange(year, month)[1]
        nstp = int(np.ceil(ndays / 5.0))
        periods.append((float(ndays), nstp, 1.0))

sim_length_days = sum(p[0] for p in periods)
total_steps = sum(int(p[1]) for p in periods)
dt = 5.0  # target time step [days]

# Courant number estimates
v_avg = 2.3     # m/d (domain-average seepage velocity from NB2)
v_river = 11.0  # m/d (near-river velocity, e.g. near Hardhof wellfield)
dx_estimate = 25.0  # m (river cell size)
Cr_avg = v_avg * dt / dx_estimate
Cr_river = v_river * dt / dx_estimate

print(f'Stress periods: {nper} monthly periods ({year_start}–{year_end})')
print(f'Total simulation: {sim_length_days:.0f} days ({sim_length_days/365.25:.1f} years)')
print(f'Total time steps: {total_steps} (~{dt:.0f}-day steps)')
print(f'\nCourant number estimates:')
print(f'  Domain-average: Cr = {v_avg} x {dt} / {dx_estimate} = {Cr_avg:.2f}')
print(f'  Near-river:     Cr = {v_river} x {dt} / {dx_estimate} = {Cr_river:.1f}')
print(f'  → Cr > 1 near rivers — using TVD advection scheme for stability')

In [ ]:
# Checkpoint 2 — Courant Number
check_task_with_solution('task_t04_checkpoint_2')

---

## 5 — Temperature Boundary Conditions

For SSM coupling, each GWF stress package needs an `auxiliary=['TEMPERATURE']` column. The SSM package in GWE then reads these temperatures automatically — every unit of water that enters or leaves the model carries its assigned temperature.

The key step is **classifying river cells as Limmat or Sihl**, since they have different monthly temperature series. We build a boolean mask (`is_limmat`) that we reuse when constructing transient stress period data.

In [ ]:
# Classify river cells as Limmat or Sihl
rivers_path = download_named_file(name='rivers', data_type='gis')
rivers_gdf = gpd.read_file(rivers_path)
rivers_gdf = rivers_gdf[rivers_gdf['GEWAESSERNAME'].isin(['Sihl', 'Limmat'])]

limmat_poly = rivers_gdf[rivers_gdf['GEWAESSERNAME'] == 'Limmat'].union_all()
sihl_poly = rivers_gdf[rivers_gdf['GEWAESSERNAME'] == 'Sihl'].union_all()

# Get cell centres
xc = modelgrid.xcellcenters
yc = modelgrid.ycellcenters

# Classify each RIV cell: point-in-polygon, fallback to nearest distance
is_limmat = []  # Boolean: True = Limmat, False = Sihl
riv_temperatures = []  # Annual mean (for reference / checkpoint 3)
n_limmat = 0
n_sihl = 0

for entry in riv_spd_raw:
    cell_id = entry['cellid']
    cell_idx = cell_id[1] if isinstance(cell_id, tuple) else cell_id
    pt = Point(xc[cell_idx], yc[cell_idx])

    if limmat_poly.contains(pt):
        is_limmat.append(True)
        riv_temperatures.append(T_limmat)
        n_limmat += 1
    elif sihl_poly.contains(pt):
        is_limmat.append(False)
        riv_temperatures.append(T_sihl)
        n_sihl += 1
    else:
        # Fallback: assign based on nearest river polygon
        d_limmat = pt.distance(limmat_poly)
        d_sihl = pt.distance(sihl_poly)
        if d_limmat <= d_sihl:
            is_limmat.append(True)
            riv_temperatures.append(T_limmat)
            n_limmat += 1
        else:
            is_limmat.append(False)
            riv_temperatures.append(T_sihl)
            n_sihl += 1

print(f'River cell classification:')
print(f'  Limmat: {n_limmat} cells (monthly T: {T_limmat_monthly.min():.1f}–{T_limmat_monthly.max():.1f} °C)')
print(f'  Sihl:   {n_sihl} cells (monthly T: {T_sihl_monthly.min():.1f}–{T_sihl_monthly.max():.1f} °C)')
print(f'  Total:  {len(is_limmat)} cells')

### Temperature Assignment Summary

| Boundary | Package | Temperature | Data Source |
|----------|---------|-------------|-------------|
| River Limmat | RIV (aux) | Monthly mean: 5–26 °C | Station 0578 (full record) |
| River Sihl | RIV (aux) | Monthly mean: 2–25 °C | Station 0577 (full record) |
| Areal recharge | RCHA (aux) | Monthly air T (clamped >0.5 °C) | Fluntern station (full record) |
| Lateral inflow (WEL) | WEL (aux) | 10.5 °C (constant) | Background groundwater |
| CHD outflow | CHD (aux) | 10.5 °C (constant) | Background (outflow, minor effect) |

> **Why seasonal forcing?** River temperatures show a strong annual cycle (~20 °C amplitude for the Limmat, ~23 °C for the Sihl). This drives seasonal heat pulses into the aquifer that attenuate and lag with distance from the river — a key signal for calibrating dispersivity in NB5.

In [ ]:
# Checkpoint 3 — Dominant Thermal Input
create_multiple_choice('task_t04_checkpoint_3')

<details>
<summary><strong>Quick check: energy budget reasoning</strong> (click to expand)</summary>

The thermal power anomaly from each source is $\Phi = \rho_w c_w Q \Delta T$, where $\Delta T = T_{source} - T_{background}$.

The Limmat has both the largest flux ($Q$) and the largest temperature anomaly ($\Delta T = +2$ °C), making it the dominant thermal input by a wide margin. Areal recharge covers a large area but its anomaly ($-0.7$ °C) is small, and the Sihl is near background temperature ($+0.6$ °C).

This is consistent with the energy budget analysis from NB2.

</details>

---

## 6 — Build Coupled GWF + GWE Simulation

We now build the coupled simulation from scratch. The process has five steps:

1. **Create simulation** with monthly time discretisation
2. **Build GWF** with temperature auxiliaries on all stress packages
3. **Build GWE** with thermal parameters matching our perceptual model
4. **Configure solvers** — separate IMS for each model
5. **Link models** via the GWF-GWE exchange [3, 4]

In [ ]:
# Step 1: Create simulation and time discretisation
# (Skipped if cached results exist — see FORCE_RERUN flag in cell above)
ucn_cached = os.path.join(TRANSPORT_WS, 'gwe_limmat.ucn')
_need_build = FORCE_RERUN or not os.path.exists(ucn_cached)

if _need_build:
    # Step 1: Create simulation and time discretisation
    sim = MFSimulation(sim_name='transport_model', sim_ws=TRANSPORT_WS)
    ModflowTdis(sim, nper=nper, perioddata=periods)
    print(f'Simulation created: {nper} stress periods in {TRANSPORT_WS}')
else:
    print("Build skipped — using cached simulation results")


In [ ]:
if _need_build:
    # Step 2: Build GWF model with temperature auxiliaries
    gwf = ModflowGwf(sim, modelname='gwf_limmat', save_flows=True,
                     newtonoptions='NEWTON')

    # DISV grid
    ModflowGwfdisv(gwf, nlay=nlay, ncpl=ncpl,
                   nvert=gridprops['nvert'], vertices=gridprops['vertices'],
                   cell2d=gridprops['cell2d'], top=top_1d, botm=botm_layers,
                   idomain=idomain_nd)

    # Hydraulic properties — convertible top layer, confined below; Kv = K/10
    icelltype = [1] + [0] * (nlay - 1)
    ModflowGwfnpf(gwf, icelltype=icelltype, k=k_3d, k33=k33,
                  save_flows=True, save_specific_discharge=True)

    # Initial heads from solved flow model (replicated across layers)
    ModflowGwfic(gwf, strt=initial_heads_3d)

    # --- Transient RIV: monthly temperatures, layer 0 ONLY ---
    # River infiltrate enters the top layer of the aquifer.
    riv_spd_transient = {}
    for per in range(nper):
        T_lim = T_limmat_monthly[per]
        T_sih = T_sihl_monthly[per]
        riv_per = []
        for i, entry in enumerate(riv_spd_raw):
            T_riv = T_lim if is_limmat[i] else T_sih
            # Force layer 0: cellid = (0, cell_index)
            cell_id = entry['cellid']
            cell_idx = cell_id[1] if isinstance(cell_id, tuple) else cell_id
            riv_per.append([(0, cell_idx), entry['stage'], entry['cond'], entry['rbot'], T_riv])
        riv_spd_transient[per] = riv_per
    ModflowGwfriv(gwf, stress_period_data=riv_spd_transient,
                  auxiliary=['TEMPERATURE'], pname='RIV-1')

    # --- CHD: assign to layers where head >= cell bottom ---
    chd_spd_with_temp = []
    for entry in chd_spd_raw:
        cell_id = entry['cellid']
        cell_idx = cell_id[1] if isinstance(cell_id, tuple) else cell_id
        head_val = entry['head']
        for lay in range(nlay):
            cell_bot = botm_layers[lay, cell_idx]
            if head_val >= cell_bot:
                chd_spd_with_temp.append([(lay, cell_idx), head_val, T_chd])
    ModflowGwfchd(gwf, stress_period_data={0: chd_spd_with_temp},
                  auxiliary=['TEMPERATURE'], pname='CHD-1')

    # --- WEL: distribute by layer thickness ---
    wel_spd_with_temp = []
    for entry in wel_spd_raw:
        cell_id = entry['cellid']
        cell_idx = cell_id[1] if isinstance(cell_id, tuple) else cell_id
        q_total = entry['q']
        # Compute layer thicknesses at this cell
        lay_thick = np.zeros(nlay)
        for lay in range(nlay):
            upper = top_1d[cell_idx] if lay == 0 else botm_layers[lay - 1, cell_idx]
            lay_thick[lay] = upper - botm_layers[lay, cell_idx]
        wet = lay_thick > 0.1
        if wet.any():
            frac = lay_thick * wet / (lay_thick * wet).sum()
            for lay in range(nlay):
                if frac[lay] > 0.01:
                    wel_spd_with_temp.append([(lay, cell_idx), q_total * frac[lay], T_lateral])
        else:
            wel_spd_with_temp.append([(0, cell_idx), q_total, T_lateral])
    ModflowGwfwel(gwf, stress_period_data={0: wel_spd_with_temp},
                  auxiliary=['TEMPERATURE'], pname='WEL-1')

    # --- Transient RCHA: monthly recharge temperatures (enters layer 0 automatically) ---
    rcha_aux_transient = {}
    for per in range(nper):
        temp_arr = np.where(recharge_array > 0, T_recharge_monthly[per], 0.0)
        rcha_aux_transient[per] = [temp_arr]
    ModflowGwfrcha(gwf, recharge=recharge_array,
                   auxiliary='TEMPERATURE', aux=rcha_aux_transient,
                   pname='RCHA-1')

    # Output control: save heads at end of each stress period
    ModflowGwfoc(gwf,
                 head_filerecord='gwf_limmat.hds',
                 budget_filerecord='gwf_limmat.cbc',
                 saverecord=[('HEAD', 'LAST'),
                             ('BUDGET', 'ALL')])

    print(f'GWF model built with {nlay} layer(s) and transient temperature forcing:')
    print(f'  DISV: {nlay} x {ncpl} = {nlay * ncpl} cells')
    print(f'  NPF: icelltype={icelltype}, K={k_1d.min():.1f}-{k_1d.max():.1f} m/d, Kv=K/10')
    print(f'  RIV: {len(riv_spd_raw)} cells x {nper} periods (layer 0 only)')
    print(f'    Limmat: {n_limmat} cells')
    print(f'    Sihl:   {n_sihl} cells')
    print(f'  CHD: {len(chd_spd_with_temp)} cell-layer entries at {T_chd} °C')
    print(f'  WEL: {len(wel_spd_with_temp)} cell-layer entries at {T_lateral} °C')
    print(f'  RCHA: monthly recharge T ({T_recharge_monthly.min():.1f}–{T_recharge_monthly.max():.1f} °C)')


In [ ]:
if _need_build:
    # ============================================================
    # ESL — Compute cell areas and build stress period data
    # ============================================================
    from shapely.geometry import Polygon as ShapelyPolygon

    # Compute DISV cell areas and centroids from vertex coordinates
    verts_data = gridprops['vertices']
    cell2d_data = gridprops['cell2d']

    # Build vertex lookup: vertex_id -> (x, y)
    vert_dict = {}
    for v in verts_data:
        vert_dict[v[0]] = (v[1], v[2])

    cell_areas = np.zeros(ncpl)
    cell_xc = np.zeros(ncpl)
    cell_yc = np.zeros(ncpl)
    for rec in cell2d_data:
        icell = rec[0]
        cell_xc[icell] = rec[1]
        cell_yc[icell] = rec[2]
        nv = rec[3]
        vert_ids = rec[4:4+nv]
        poly_coords = [vert_dict[vid] for vid in vert_ids]
        cell_areas[icell] = ShapelyPolygon(poly_coords).area

    # Build combined ESL stress period data
    # 1) Geothermal: bottom layer (nlay-1), all active cells
    # 2) Urban heat: top layer (0), active cells where xc > urban_x_min
    esl_spd = []
    active_flat = idomain_nd.copy()  # (nlay, ncpl)

    # Geothermal on bottom layer
    n_geo = 0
    P_geo_total = 0.0
    bot_lay = nlay - 1
    for icell in range(ncpl):
        if active_flat[bot_lay, icell] > 0:
            rate = q_geo_daily * cell_areas[icell]
            esl_spd.append([(bot_lay, icell), rate])
            n_geo += 1
            P_geo_total += rate

    # Urban heat on top layer
    n_urban = 0
    P_urban_total = 0.0
    for icell in range(ncpl):
        if active_flat[0, icell] > 0 and cell_xc[icell] > urban_x_min:
            rate = q_urban_daily * cell_areas[icell]
            esl_spd.append([(0, icell), rate])
            n_urban += 1
            P_urban_total += rate

    # Convert from J/d to kW for reporting: 1 kW = 86400000 J/d
    P_geo_kW = P_geo_total / 86_400_000
    P_urban_kW = P_urban_total / 86_400_000

    print(f'ESL stress period data:')
    print(f'  Geothermal (layer {bot_lay}): {n_geo} cells, {P_geo_kW:.0f} kW total')
    print(f'  Urban SUHI (layer 0):  {n_urban} cells, {P_urban_kW:.0f} kW total')
    print(f'  Combined: {len(esl_spd)} entries')


In [ ]:
if _need_build:
    # Step 3: Build GWE model (matching GWF grid)
    gwe = ModflowGwe(sim, modelname='gwe_limmat')

    # GWE needs its own DISV (same grid as GWF)
    ModflowGwedisv(gwe, nlay=nlay, ncpl=ncpl,
                   nvert=gridprops['nvert'], vertices=gridprops['vertices'],
                   cell2d=gridprops['cell2d'], top=top_1d, botm=botm_layers,
                   idomain=idomain_nd)

    # EST — Energy Storage and Transfer
    ModflowGweest(gwe, porosity=n_e,
                  density_solid=rho_s, heat_capacity_solid=c_s,
                  density_water=rho_w, heat_capacity_water=c_w,
                  save_flows=True)

    # CND — Conduction and Mechanical Dispersion (with vertical dispersivity)
    ModflowGwecnd(gwe, alh=alpha_L, ath1=alpha_T, atv=alpha_T,
                  ktw=ktw, kts=kts)

    # ADV — Advection (TVD for stability — local Cr can exceed 1 near rivers)
    ModflowGweadv(gwe, scheme='TVD')

    # SSM — Source-Sink Mixing (reads temperature from GWF auxiliaries)
    ModflowGwessm(gwe, save_flows=True, sources=[
        ('RIV-1', 'AUX', 'TEMPERATURE'),
        ('CHD-1', 'AUX', 'TEMPERATURE'),
        ('WEL-1', 'AUX', 'TEMPERATURE'),
        ('RCHA-1', 'AUX', 'TEMPERATURE'),
    ])

    # ESL — Energy Source Loading (geothermal + urban heat)
    ModflowGweesl(gwe, stress_period_data={0: esl_spd}, save_flows=True)

    # IC — Initial temperature (uniform background across all layers)
    ModflowGweic(gwe, strt=T_background)

    # OC — Output control: save temperature at end of each stress period
    ModflowGweoc(gwe,
                 temperature_filerecord='gwe_limmat.ucn',
                 budget_filerecord='gwe_limmat.cbc',
                 saverecord=[('TEMPERATURE', 'LAST'),
                             ('BUDGET', 'ALL')])

    print(f'GWE model built ({nlay} layers):')
    print(f'  EST: n_e={n_e}, rho_s={rho_s}, c_s={c_s}')
    print(f'  CND: alpha_L={alpha_L} m, alpha_T={alpha_T} m, alpha_V={alpha_T} m, ktw={ktw:.0f}, kts={kts:.0f} J/(d*m*K)')
    print(f'  ADV: TVD')
    print(f'  SSM: RIV-1, CHD-1, WEL-1, RCHA-1')
    print(f'  ESL: geothermal ({P_geo_kW:.0f} kW) + urban SUHI ({P_urban_kW:.0f} kW)')
    print(f'  IC:  {T_background} °C (uniform, all layers)')
    print(f'  OC:  save LAST per period ({nper} snapshots)')


In [ ]:
if _need_build:
    # Step 4: Solvers

    # IMS for GWF (COMPLEX — Newton solver needs tighter control)
    ims_gwf = ModflowIms(
        sim, complexity='COMPLEX',
        outer_maximum=500, inner_maximum=300,
        outer_dvclose=1e-2, inner_dvclose=1e-4,
        linear_acceleration='BICGSTAB',
        under_relaxation='DBD',
        under_relaxation_theta=0.7,
        under_relaxation_kappa=0.1,
        under_relaxation_gamma=0.0,
        filename='gwf.ims'
    )
    sim.register_ims_package(ims_gwf, [gwf.name])

    # IMS for GWE (MODERATE — linear transport)
    ims_gwe = ModflowIms(
        sim, complexity='MODERATE',
        outer_maximum=200, inner_maximum=100,
        outer_dvclose=1e-4, inner_dvclose=1e-7,
        linear_acceleration='BICGSTAB',
        filename='gwe.ims'
    )
    sim.register_ims_package(ims_gwe, [gwe.name])

    # Step 5: GWF-GWE exchange
    ModflowGwfgwe(
        sim,
        exgtype='GWF6-GWE6',
        exgmnamea=gwf.name,
        exgmnameb=gwe.name,
    )

    print('Solvers and exchange configured:')
    print(f'  GWF solver: COMPLEX (Newton, BICGSTAB)')
    print(f'  GWE solver: MODERATE (BICGSTAB, outer_max=200, inner_dvclose=1e-7)')
    print(f'  Exchange: GWF6-GWE6 ({gwf.name} ↔ {gwe.name})')


<details>
<summary><strong>Optional: GWT (solute) equivalent code</strong> (click to expand)</summary>

For comparison, here is how the GWE code above maps to GWT (solute transport):

```python
# GWE (heat)                           # GWT (solute)
ModflowGweest(gwe,                     # ModflowGwtmst(gwt,
    porosity=n_e,                       #     porosity=n_e,
    density_solid=rho_s,                #     sorption='linear',
    heat_capacity_solid=c_s,            #     bulk_density=rho_b,
    density_water=rho_w,                #     distcoef=Kd,
    heat_capacity_water=c_w)            #     )

ModflowGwecnd(gwe,                     # ModflowGwtdsp(gwt,
    alh=alpha_L, ath1=alpha_T,          #     alh=alpha_L, ath1=alpha_T,
    ktw=ktw, kts=kts)                   #     diffc=Dm_star)

ModflowGweadv(gwe, scheme='TVD')       # ModflowGwtadv(gwt, scheme='TVD')
ModflowGwessm(gwe, sources=[...])      # ModflowGwtssm(gwt, sources=[...])
ModflowGweic(gwe, strt=T_bg)           # ModflowGwtic(gwt, strt=C_bg)
```

Key differences: EST uses thermal properties (density, heat capacity) while MST uses sorption parameters; CND uses thermal conductivity while DSP uses molecular diffusion. **Dispersivities are identical.**

</details>

> **Why two solvers?** The GWF model uses Newton-Raphson (nonlinear, for unconfined flow) while the GWE transport equation is linear — a simpler solver suffices. Each model gets its own IMS instance registered separately.

In [ ]:
# Write and run simulation — skip if cached results exist
ucn_cached = os.path.join(TRANSPORT_WS, 'gwe_limmat.ucn')

if not FORCE_RERUN and os.path.exists(ucn_cached):
    print(f'Cached results found — skipping simulation (set FORCE_RERUN = True to re-run)')
    print(f'  {ucn_cached}')
    success = True
else:
    sim.write_simulation()
    print(f'Running transient GWF+GWE simulation ({nper} stress periods, {total_steps} time steps)...')
    print('This may take several minutes...')
    success, buff = sim.run_simulation(silent=True)

    if success:
        print('Simulation completed successfully!')
    else:
        print(f'Simulation FAILED. Check: {os.path.join(TRANSPORT_WS, "mfsim.lst")}')
        lst_path = os.path.join(TRANSPORT_WS, 'mfsim.lst')
        if os.path.exists(lst_path):
            with open(lst_path) as f:
                lines = f.readlines()
            print('Last 20 lines of mfsim.lst:')
            for line in lines[-20:]:
                print(f'  {line.rstrip()}')

<details>
<summary><strong>Limitation: RIV+SSM only captures advective heat exchange</strong> (click to expand)</summary>

The RIV+SSM coupling in MODFLOW 6 only handles **advective** heat transfer — heat carried by water flowing through the riverbed. Three distinct flux regimes:

| Regime | Flow direction | Heat transfer captured | What’s missing |
|---|---|---|---|
| **Losing** (stage > head) | River → aquifer | SSM applies river temperature to infiltrating water. Advective + dispersive fluxes act in same direction. | **Well-represented** |
| **Gaining** (head > stage) | Aquifer → river | SSM applies aquifer temperature to exfiltrating water. | Diffusive/conductive heat from warm river into aquifer (counter to flow) is **not captured** |
| **No net flow** (head ≈ stage) | ~Zero | SSM transfers no heat ($Q_{riv} \approx 0$). | **Conductive heat exchange through the riverbed is completely missing** |

This simplification:
- **Underestimates** heat exchange in gaining reaches and near-stagnant zones
- Is **acceptable** for the Limmat Valley where losing reaches dominate the thermal signal
- Could be improved with the SFR (Streamflow Routing) package or finer vertical discretisation near the riverbed [2], but that is beyond scope for this course

</details>

---

## 7 — Results: Seasonal Temperature Dynamics

With transient forcing, we expect to see the seasonal temperature signal propagate from the rivers into the aquifer — warm in summer, cool in winter. The signal should **attenuate** and **lag** with distance from the river, controlled by thermal retardation and dispersivity.

In [ ]:
# Load temperature results
ucn_path = os.path.join(TRANSPORT_WS, 'gwe_limmat.ucn')
temp_file = flopy.utils.HeadFile(ucn_path, text='TEMPERATURE')
times = temp_file.get_times()
kstpkper = temp_file.get_kstpkper()

# Select 4 seasonal snapshots from the last years of the simulation
# Period index = (year - year_start) * 12 + (month - 1)
snapshot_specs = [
    (year_end - 1, 1, 'Winter'),   # January, second-to-last year
    (year_end - 1, 7, 'Summer'),   # July, second-to-last year
    (year_end, 1, 'Winter'),       # January, last year
    (year_end, 7, 'Summer'),       # July, last year
]
seasonal_periods = []
seasonal_labels = []
for y, m, season in snapshot_specs:
    per = common_years.index(y) * 12 + (m - 1)
    seasonal_periods.append(per)
    month_name = 'Jan' if m == 1 else 'Jul'
    seasonal_labels.append(f'{season} {y}\n({month_name})')

seasonal_data = [temp_file.get_data(kstpkper=kstpkper[per])[0].flatten() for per in seasonal_periods]

print(f'Temperature output: {len(times)} saved time steps')
print(f'Time range: {times[0]:.0f} to {times[-1]:.0f} days ({times[-1]/365.25:.1f} years)')
print(f'\nSeasonal snapshots (after {nyears} years of forcing):')
for label, per, data in zip(seasonal_labels, seasonal_periods, seasonal_data):
    label_short = label.replace('\n', ' ')
    print(f'  {label_short} (per={per}): T = {data.min():.1f} to {data.max():.1f} °C, mean = {data.mean():.1f} °C')

In [ ]:
# Plot 2x2 seasonal temperature maps
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# Shared colour limits across all seasonal snapshots
vmin = min(d.min() for d in seasonal_data)
vmax = max(d.max() for d in seasonal_data)

for ax, label, data in zip(axes.flatten(), seasonal_labels, seasonal_data):
    quick_model_plot(
        modelgrid, data, boundary_gdf=boundary_gdf,
        title=label, cmap='RdYlBu_r',
        colorbar_label='Temperature (°C)',
        vmin=vmin, vmax=vmax, ax=ax
    )

plt.suptitle('Seasonal Temperature Snapshots — Transient Heat Transport', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Temperature time series at 3 monitoring points — distance from Sihl
# We track from the Sihl to show attenuation with distance from a river boundary.
active_mask = idomain.flatten() > 0

# Find Sihl river cells and pick one in the middle reach
sihl_riv_indices = [i for i, flag in enumerate(is_limmat) if not flag]
mid_sihl_idx = len(sihl_riv_indices) // 2
sihl_cell_id = riv_spd_raw[sihl_riv_indices[mid_sihl_idx]]['cellid']
sihl_cell = sihl_cell_id[1] if isinstance(sihl_cell_id, tuple) else sihl_cell_id

# Compute distance from this mid-Sihl cell to all grid cells
sihl_x, sihl_y = xc[sihl_cell], yc[sihl_cell]
dists_from_sihl = np.sqrt((xc - sihl_x)**2 + (yc - sihl_y)**2)
dists_from_sihl[~active_mask] = 1e9

# Near Sihl: 50–150 m away
near_mask = (dists_from_sihl > 50) & (dists_from_sihl < 150)
if near_mask.any():
    near_cell = np.where(near_mask)[0][np.argmin(dists_from_sihl[near_mask])]
else:
    near_cell = np.argmin(dists_from_sihl[dists_from_sihl > 20])

# Mid-aquifer: 200–400 m from the Sihl
mid_mask = (dists_from_sihl > 200) & (dists_from_sihl < 400) & active_mask
if mid_mask.any():
    mid_cell = np.where(mid_mask)[0][np.argmin(dists_from_sihl[mid_mask])]
else:
    mid_cell = np.argmin(np.abs(dists_from_sihl - 300))

# Far from Sihl: >600 m
far_mask = (dists_from_sihl > 600) & active_mask
if far_mask.any():
    far_cell = np.where(far_mask)[0][0]
else:
    far_cell = np.argmax(dists_from_sihl[active_mask])

monitor_cells = [near_cell, mid_cell, far_cell]
monitor_labels = [
    f'Near Sihl (~{dists_from_sihl[near_cell]:.0f} m)',
    f'Mid-aquifer (~{dists_from_sihl[mid_cell]:.0f} m)',
    f'Far from Sihl (~{dists_from_sihl[far_cell]:.0f} m)'
]

# Extract time series at each monitoring point
monitor_ts = []
for cell in monitor_cells:
    ts = [temp_file.get_data(kstpkper=kstpkper[per])[0].flatten()[cell] for per in range(nper)]
    monitor_ts.append(ts)

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['tab:red', 'tab:orange', 'tab:blue']
for ts, label, color in zip(monitor_ts, monitor_labels, colors):
    ax.plot(month_labels, ts, '-', color=color, label=label, linewidth=1.5)

# Also plot Sihl forcing for reference (thin line)
ax.plot(month_labels, T_sihl_monthly, '--', color='gray', alpha=0.5, linewidth=1, label='Sihl river T')
ax.axhline(T_background, color='gray', linestyle=':', alpha=0.4)
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Temperature Time Series — Attenuation with Distance from Sihl')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('The seasonal signal attenuates and lags with distance from the river.')
print('Near the river: larger amplitude, closer to river forcing.')
print('Far from river: damped signal, approaches background temperature.')
if nlay > 1:
    print(f'The {nlay}-layer model concentrates river infiltrate in the top layer, producing stronger seasonal amplitudes.')
else:
    print('With a single layer, the seasonal signal is diluted over the full aquifer thickness. Try nlay=3 for vertical resolution.')

In [ ]:
# Temperature statistics — summer vs winter comparison
active = idomain.flatten() > 0

# Summer (last Jul) and Winter (last Jan) from the snapshots above
T_summer = seasonal_data[3]  # last summer snapshot
T_winter = seasonal_data[2]  # last winter snapshot
T_final = temp_file.get_data(kstpkper=kstpkper[-1])[0].flatten()  # Last period

summer_label = seasonal_labels[3].replace('\n', ' ')
winter_label = seasonal_labels[2].replace('\n', ' ')

print(f'Temperature statistics — seasonal comparison:')
print(f'  {"":20s} {winter_label:>15s} {summer_label:>15s} {"Final":>15s}')
print(f'  {"Min":20s} {T_winter[active].min():>15.1f} {T_summer[active].min():>15.1f} {T_final[active].min():>15.1f} °C')
print(f'  {"Max":20s} {T_winter[active].max():>15.1f} {T_summer[active].max():>15.1f} {T_final[active].max():>15.1f} °C')
print(f'  {"Mean":20s} {T_winter[active].mean():>15.2f} {T_summer[active].mean():>15.2f} {T_final[active].mean():>15.2f} °C')
print(f'  {"Std":20s} {T_winter[active].std():>15.2f} {T_summer[active].std():>15.2f} {T_final[active].std():>15.2f} °C')
print(f'  {"Cells > 15 °C":20s} {(T_winter[active] > 15).sum():>15d} {(T_summer[active] > 15).sum():>15d} {(T_final[active] > 15).sum():>15d}')

T_active = T_summer[active]  # For checkpoint 4

In [ ]:
# Checkpoint — Seasonal Signal Attenuation
create_multiple_choice('task_t04_checkpoint_seasonal')

### Note: Warming Trend in Results

Later years in the simulation are systematically warmer than earlier years. This is **not a model artifact** — it reflects the real warming trend in the forcing data (~+0.4 °C/decade, established in NB2). The transient monthly forcing faithfully propagates this climate signal through the aquifer.

In [ ]:
# Checkpoint 4 — Mean Aquifer Temperature in last summer snapshot
check_task_with_solution('task_t04_checkpoint_4')

### Reflection: Vertical Resolution in Heat Transport

With `nlay = 1` (single-layer, the default), the model distributes all fluxes uniformly over the aquifer thickness. This is adequate for capturing domain-average thermal dynamics but dilutes the concentrated near-surface signal from river infiltrate.

**Why might vertical resolution matter?**

In reality, warm river infiltrate forms a thin plume in the upper few metres of the aquifer. A single-layer model dilutes this signal over the full aquifer thickness (~20–40 m), reducing the seasonal amplitude. With `nlay = 3` (15% / 30% / 55% of thickness), the top layer captures the concentrated thermal signal while deeper layers remain closer to background temperature.

| | Single-layer (diluted) | 3-layer (top layer) |
|---|---|---|
| **Near a losing reach in summer** | Column average: ~11–13 °C | Top layer: 15–18 °C |
| **Seasonal swing at 100 m from river** | ~0.5–1 °C | ~2–5 °C |
| **Thermal plume extent** | Narrow ribbon along river | Wider, physically realistic spread |

**The mean temperature is still governed by flux-weighted mixing.** The Limmat's annual mean (12.5 °C) is only 2 °C above background, and the Sihl (11.1 °C) is barely warmer. After mixing with recharge (~10 °C) and lateral inflow (10.5 °C), a modest domain-average warming is expected regardless of vertical resolution.

> **Key insight:** For quantitative calibration against monitoring wells that sample the upper aquifer, vertical resolution can improve the match. Try changing `nlay = 3` in Section 2.1 to explore this effect.

### Energy Balance

The energy budget shows where heat enters and leaves the aquifer. Look for which component dominates — this should be consistent with our perceptual model from NB2. A low percent error (<1%) confirms the numerical solution conserves energy.

In [ ]:
# Read GWE energy budget
try:
    cbc_path = os.path.join(TRANSPORT_WS, 'gwe_limmat.cbc')
    budget = flopy.utils.CellBudgetFile(cbc_path)
    record_names = budget.get_unique_record_names()
    budget_kstpkper = budget.get_kstpkper()
    last_kstpkper = budget_kstpkper[-1]

    print('GWE Energy Budget (last saved time step):')
    print(f'{"Component":<25s} {"Inflow":>15s} {"Outflow":>15s} {"Net":>15s}')
    print('-' * 70)

    total_in = 0.0
    total_out = 0.0

    for name in record_names:
        # Record names may be bytes — decode to string
        name_str = name.decode().strip() if isinstance(name, bytes) else str(name).strip()
        data = budget.get_data(text=name, kstpkper=last_kstpkper)
        if len(data) > 0:
            d = data[0]
            # GWE budget records may be arrays or structured arrays
            if hasattr(d, 'dtype') and d.dtype.names and 'q' in d.dtype.names:
                vals = d['q']
            else:
                vals = d.flatten()
            inflow = float(vals[vals > 0].sum())
            outflow = float(vals[vals < 0].sum())
            net = inflow + outflow
            total_in += inflow
            total_out += abs(outflow)
            if abs(inflow) > 0 or abs(outflow) > 0:
                print(f'{name_str:<25s} {inflow:>15.1f} {outflow:>15.1f} {net:>15.1f}')

    print('-' * 70)
    print(f'{"TOTAL":<25s} {total_in:>15.1f} {-total_out:>15.1f} {total_in - total_out:>15.1f}')
    if (total_in + total_out) > 0:
        pct_error = 200 * abs(total_in - total_out) / (total_in + total_out)
        print(f'{"PERCENT ERROR":<25s} {pct_error:>15.4f}%')

except Exception as e:
    print(f'Could not read GWE budget: {e}')
    print('The model may not have converged or budget output may not be saved.')

In [ ]:
# Checkpoint 5 — Energy Budget
create_multiple_choice('task_t04_checkpoint_5')

---

## 8 — Sensitivity Exercise: Dispersivity

Longitudinal dispersivity ($\alpha_L$) controls how much the thermal signal spreads as it moves through the aquifer. With transient forcing, dispersivity also controls how much the **seasonal amplitude is damped** — higher dispersivity means more mixing, which smooths out the seasonal peaks.

> **Exercise:** Change `alpha_test` below to different values (e.g., 1, 5, 50 m) and observe how the summer temperature plume changes. The cell rebuilds the GWE model in a separate subdirectory and compares the summer 2023 snapshot to the base case.

In [ ]:
# ============================================================
# SENSITIVITY EXERCISE: Change alpha_test and rerun this cell
# ============================================================
alpha_test = 50.0  # <-- CHANGE THIS VALUE (try: 1, 5, 10, 50, 100)

# Reconstruct boundary condition data if using cached results
if 'icelltype' not in dir():
    icelltype = [1] + [0] * (nlay - 1)
if 'riv_spd_transient' not in dir():
    riv_spd_transient = {}
    for per in range(nper):
        T_lim = T_limmat_monthly[per]
        T_sih = T_sihl_monthly[per]
        riv_per = []
        for i, entry in enumerate(riv_spd_raw):
            T_riv = T_lim if is_limmat[i] else T_sih
            cell_id = entry['cellid']
            cell_idx = cell_id[1] if isinstance(cell_id, tuple) else cell_id
            riv_per.append([(0, cell_idx), entry['stage'], entry['cond'], entry['rbot'], T_riv])
        riv_spd_transient[per] = riv_per
if 'chd_spd_with_temp' not in dir():
    chd_spd_with_temp = []
    for entry in chd_spd_raw:
        cell_id = entry['cellid']
        cell_idx = cell_id[1] if isinstance(cell_id, tuple) else cell_id
        for lay in range(nlay):
            upper = top_1d[cell_idx] if lay == 0 else botm_layers[lay - 1, cell_idx]
            if entry['head'] >= botm_layers[lay, cell_idx]:
                chd_spd_with_temp.append([(lay, cell_idx), entry['head'], T_chd])
if 'wel_spd_with_temp' not in dir():
    wel_spd_with_temp = []
    for entry in wel_spd_raw:
        cell_idx = entry['cellid'][1] if isinstance(entry['cellid'], tuple) else entry['cellid']
        q_total = entry['q']
        lay_thick = np.zeros(nlay)
        for lay in range(nlay):
            upper = top_1d[cell_idx] if lay == 0 else botm_layers[lay - 1, cell_idx]
            lay_thick[lay] = upper - botm_layers[lay, cell_idx]
        wet = lay_thick > 0.1
        if wet.any():
            frac = lay_thick * wet / (lay_thick * wet).sum()
            for lay in range(nlay):
                if frac[lay] > 0.01:
                    wel_spd_with_temp.append([(lay, cell_idx), q_total * frac[lay], T_lateral])
        else:
            wel_spd_with_temp.append([(0, cell_idx), q_total, T_lateral])
if 'rcha_aux_transient' not in dir():
    rcha_aux_transient = {}
    for per in range(nper):
        temp_arr = np.where(recharge_array > 0, T_recharge_monthly[per], 0.0)
        rcha_aux_transient[per] = [temp_arr]

# Create sensitivity workspace
SENS_WS = os.path.join(TRANSPORT_WS, 'sensitivity_test')
if os.path.exists(SENS_WS):
    shutil.rmtree(SENS_WS)
os.makedirs(SENS_WS)

# Build a fresh simulation with same transient forcing but modified dispersivity
sim_s = MFSimulation(sim_name='sens_test', sim_ws=SENS_WS)
ModflowTdis(sim_s, nper=nper, perioddata=periods)

# GWF (identical to base — reuse transient data)
gwf_s = ModflowGwf(sim_s, modelname='gwf_s', save_flows=True, newtonoptions='NEWTON')
ModflowGwfdisv(gwf_s, nlay=nlay, ncpl=ncpl, nvert=gridprops['nvert'],
               vertices=gridprops['vertices'], cell2d=gridprops['cell2d'],
               top=top_1d, botm=botm_layers, idomain=idomain_nd)
ModflowGwfnpf(gwf_s, icelltype=icelltype, k=k_3d, k33=k33,
              save_flows=True, save_specific_discharge=True)
ModflowGwfic(gwf_s, strt=initial_heads_3d)
ModflowGwfriv(gwf_s, stress_period_data=riv_spd_transient,
              auxiliary=['TEMPERATURE'], pname='RIV-1')
ModflowGwfchd(gwf_s, stress_period_data={0: chd_spd_with_temp},
              auxiliary=['TEMPERATURE'], pname='CHD-1')
ModflowGwfwel(gwf_s, stress_period_data={0: wel_spd_with_temp},
              auxiliary=['TEMPERATURE'], pname='WEL-1')
ModflowGwfrcha(gwf_s, recharge=recharge_array,
               auxiliary='TEMPERATURE', aux=rcha_aux_transient,
               pname='RCHA-1')
ModflowGwfoc(gwf_s, head_filerecord='gwf_s.hds', budget_filerecord='gwf_s.cbc',
             saverecord=[('HEAD', 'LAST'), ('BUDGET', 'ALL')])

# GWE with modified dispersivity
gwe_s = ModflowGwe(sim_s, modelname='gwe_s')
ModflowGwedisv(gwe_s, nlay=nlay, ncpl=ncpl, nvert=gridprops['nvert'],
               vertices=gridprops['vertices'], cell2d=gridprops['cell2d'],
               top=top_1d, botm=botm_layers, idomain=idomain_nd)
ModflowGweest(gwe_s, porosity=n_e, density_solid=rho_s, heat_capacity_solid=c_s,
              density_water=rho_w, heat_capacity_water=c_w)
ModflowGwecnd(gwe_s, alh=alpha_test, ath1=alpha_test / 10, atv=alpha_test / 10,
              ktw=ktw, kts=kts)
ModflowGweadv(gwe_s, scheme='TVD')
ModflowGwessm(gwe_s, sources=[
    ('RIV-1', 'AUX', 'TEMPERATURE'), ('CHD-1', 'AUX', 'TEMPERATURE'),
    ('WEL-1', 'AUX', 'TEMPERATURE'), ('RCHA-1', 'AUX', 'TEMPERATURE')])
ModflowGweic(gwe_s, strt=T_background)
ModflowGweoc(gwe_s, temperature_filerecord='gwe_s.ucn', budget_filerecord='gwe_s.cbc',
             saverecord=[('TEMPERATURE', 'LAST'), ('BUDGET', 'ALL')])

# ESL — Energy Source Loading (same as base model)
if 'esl_spd' in dir():
    ModflowGweesl(gwe_s, stress_period_data={0: esl_spd})

# Solvers and exchange
ims_gwf_s = ModflowIms(sim_s, complexity='COMPLEX', outer_maximum=500,
                        inner_maximum=300, outer_dvclose=1e-2, inner_dvclose=1e-4,
                        linear_acceleration='BICGSTAB', under_relaxation='DBD',
                        under_relaxation_theta=0.7, under_relaxation_kappa=0.1,
                        under_relaxation_gamma=0.0, filename='gwf_s.ims')
sim_s.register_ims_package(ims_gwf_s, [gwf_s.name])
ims_gwe_s = ModflowIms(sim_s, complexity='MODERATE', outer_maximum=200,
                        inner_maximum=100, outer_dvclose=1e-4, inner_dvclose=1e-7,
                        linear_acceleration='BICGSTAB', filename='gwe_s.ims')
sim_s.register_ims_package(ims_gwe_s, [gwe_s.name])
ModflowGwfgwe(sim_s, exgtype='GWF6-GWE6', exgmnamea=gwf_s.name, exgmnameb=gwe_s.name)

# Run
sim_s.write_simulation()
print(f'Running sensitivity test with alpha_L = {alpha_test} m ({nper} periods, {nlay} layer(s))...')
print('This may take several minutes...')
success_s, _ = sim_s.run_simulation(silent=True)

if success_s:
    # Load sensitivity result — compare last summer snapshot (layer 0 = top)
    ucn_s = flopy.utils.HeadFile(os.path.join(SENS_WS, 'gwe_s.ucn'), text='TEMPERATURE')
    kstpkper_s = ucn_s.get_kstpkper()
    summer_per = seasonal_periods[3]  # same summer period as base case
    T_sens_summer = ucn_s.get_data(kstpkper=kstpkper_s[summer_per])[0].flatten()

    # Compare with base case summer snapshot
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

    for ax, data, label in zip(axes,
                                [T_summer, T_sens_summer],
                                [f'Base case (alpha_L = {alpha_L} m)',
                                 f'Sensitivity (alpha_L = {alpha_test} m)']):
        quick_model_plot(modelgrid, data, boundary_gdf=boundary_gdf,
                         title=label, cmap='RdYlBu_r',
                         colorbar_label='Temperature (°C)',
                         vmin=vmin, vmax=vmax, ax=ax)

    plt.suptitle(f'Dispersivity Sensitivity — {seasonal_labels[3].replace(chr(10), " ")}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    # Statistics
    T_sens_act = T_sens_summer[active]
    print(f'\nComparison — {seasonal_labels[3].replace(chr(10), " ")} snapshot (layer 0):')
    print(f'  {"":20s} {"Base (aL="+str(alpha_L)+")":>20s} {"Test (aL="+str(alpha_test)+")":>20s}')
    print(f'  {"Mean T":20s} {T_active.mean():>20.2f} {T_sens_act.mean():>20.2f} °C')
    print(f'  {"Max T":20s} {T_active.max():>20.2f} {T_sens_act.max():>20.2f} °C')
    print(f'  {"Cells > 15°C":20s} {(T_active > 15).sum():>20d} {(T_sens_act > 15).sum():>20d}')
else:
    print(f'Sensitivity run FAILED with alpha_L = {alpha_test} m')

In [ ]:
# Checkpoint — Dispersivity Sensitivity
create_multiple_choice('task_t04_checkpoint_alpha')

> **Reflection:** The right $\alpha_L$ must reproduce observed temperature patterns. In Notebook 5 (Calibration), we will use measured borehole temperatures to find the optimal dispersivity — this is why heat is such a useful calibration tracer.

---

## 9 — Summary

In [ ]:
# Dynamic model summary
if success:
    print('=' * 60)
    print('TRANSPORT MODEL SUMMARY')
    print('=' * 60)
    print(f'  Grid:              {nlay} x {ncpl} = {nlay * ncpl} cells (DISV, {nlay} layers)')
    print(f'  K range:           {k_array.min():.1f} to {k_array.max():.1f} m/day')
    print(f'  Porosity (n_e):    {n_e}')
    print(f'  alpha_L / alpha_T: {alpha_L} / {alpha_T} m')
    print(f'  lambda_bulk:       {lambda_bulk:.2f} W/(m*K)')
    print(f'  Retardation R:     {R_thermal:.2f}')
    print(f'  T_background:      {T_background} °C')
    print(f'  Forcing:           Monthly transient ({year_start}-{year_end})')
    print(f'  Stress periods:    {nper} ({nyears} years)')
    print(f'  Simulation:        {sim_length_days/365.25:.1f} years, ~{dt:.0f}-day steps')
    print(f'  Advection scheme:  TVD (Cr ~ {Cr:.1f})')
    print(f'  Summer mean T:     {T_active.mean():.2f} °C')
    print(f'  Solver:            GWF=COMPLEX, GWE=MODERATE')
    print(f'  Status:            {"Converged" if success else "FAILED"}')
    print('=' * 60)

## What You're Taking Forward

| For Notebook 5 (Calibration) | Value/Insight |
|------------------------------|---------------|
| Working GWF+GWE model | Transient coupled flow-transport with SSM coupling |
| Temperature forcing | Monthly measured data (full record) for RIV and RCHA |
| Seasonal dynamics | Summer/winter contrast visible in maps and time series |
| Signal attenuation | Seasonal amplitude decreases with distance from river |
| Dominant thermal source | Infiltrating rivers (strongest signal where river recharges aquifer) |
| Key calibration parameter | $\alpha_L$ (dispersivity controls plume shape **and** seasonal amplitude) |
| Thermal retardation | $R \approx 3.23$ (thermal front moves ~3x slower than water) |

**Key takeaways:**
- GWE uses the **same grid and flow solution** as GWF — no separate model needed
- SSM coupling is automatic: assign temperature as an auxiliary variable on GWF stress packages
- Transient forcing with **monthly stress periods** captures the seasonal temperature cycle
- The thermal signal needs **decades** to fully develop — short simulations underestimate the thermal plume
- The seasonal signal **attenuates and lags** with distance from the river — this is key observational data for calibration
- Dispersivity controls both plume spreading **and** seasonal amplitude damping

**The big picture:** We now have a transient heat transport model driven by real temperature data. The next step is to calibrate dispersivity against observed borehole temperatures, using the seasonal signal as a calibration target.

---

## Next Steps

**Continue to:** [05t_calibration.ipynb](05t_calibration.ipynb) — Calibrate transport parameters against observed temperatures

**Review if needed:** [03t_modflow_transport.ipynb](03t_modflow_transport.ipynb) — GWE packages and stability criteria

---

**Navigation:** [< Notebook 3: MODFLOW for Heat Transport](03t_modflow_transport.ipynb) | [Notebook 5: Calibration >](05t_calibration.ipynb)

---

### Optional: Export Results

In [ ]:
# Export temperature results to GeoPackage
if success:
    from disv_grid_utils import export_grid_to_geopackage

    output_gpkg = os.path.join(TRANSPORT_WS, 'temperature_results.gpkg')
    export_grid_to_geopackage(
        modelgrid, T_final, output_path=output_gpkg,
        layer_name='temperature_final', array_name='temp_C',
        crs='EPSG:2056'
    )
    print(f'Temperature results exported to: {output_gpkg}')

## References

\[1\] FloPy Documentation: Groundwater Energy Transport (GWE). [https://flopy.readthedocs.io/](https://flopy.readthedocs.io/) (accessed 2026)

\[2\] Doppler, T., Hendricks Franssen, H.-J., Kaiser H.-P., Kuhlman U., Stauffer, F. (2007): Field evidence of a dynamic leakage coefficient for modelling river–aquifer interactions. Journal of Hydrology, Volume 347, Issues 1–2, DOI: https://doi.org/10.1016/j.jhydrol.2007.09.017.

\[3\] MODFLOW 6 Release Notes: Groundwater Energy Transport (GWE) Model. U.S. Geological Survey. [https://www.usgs.gov/software/modflow-6-usgs-modular-hydrologic-model](https://www.usgs.gov/software/modflow-6-usgs-modular-hydrologic-model) (accessed 2026)

\[4\] Langevin, C.D., et al. (2022): MODFLOW 6 Description of Input and Output. U.S. Geological Survey Techniques and Methods 6-A62.